## Monitoring and Observability for Production AI Systems

Monitoring is the practice of tracking whether a production AI system is healthy;
observability is the deeper capability to explain *why* it is or isn't —
by inspecting the internal state through logs, metrics, and traces.
In production AI, both are required: monitoring tells you something is wrong;
observability tells you where to look.

## How it actually works

Production AI systems have two distinct layers that both require monitoring:

**Layer 1 — Infrastructure health (standard, borrowed from software engineering):**
- Latency: p50, p95, p99 response times
- Throughput: requests per second
- Error rate: 4xx, 5xx responses, timeouts
- Resource utilisation: CPU, GPU, memory, disk
Tools: Prometheus, Grafana, Azure Monitor, Datadog.
This layer tells you whether your API is alive.

**Layer 2 — Model health (specific to ML, most teams underprioritise this):**
- Input data distribution: are the features arriving at serving time consistent with training distribution?
- Output distribution: is the model's prediction distribution stable over time?
- Prediction confidence: is the model increasingly uncertain? More low-confidence predictions?
- Ground truth drift: when actual outcomes become available, how does prediction accuracy compare to baseline?
This layer tells you whether your model is still correct, not just whether it is running.

**The three signals in model monitoring:**

1. **Data drift** — input feature distributions have shifted from the training baseline.
   Detected by comparing feature statistics over a rolling window against training statistics.
   Statistical tests: KS test (continuous features), chi-squared test (categorical features),
   Population Stability Index (PSI) for both.

2. **Concept drift** — the relationship between features and the target has changed.
   The model's inputs look similar but the correct outputs have changed.
   Harder to detect without ground truth labels.
   Requires a labeling pipeline to continuously validate predictions.

3. **Model degradation** — accuracy, precision, recall, or business-specific metrics
   are trending downward over time.
   Requires ground truth labels, which are often delayed.
   Design a shadow evaluation pipeline: collect predictions, wait for true outcomes,
   compute metrics in batch, alert when below threshold.

**Observability in AI systems:**
Beyond aggregated metrics, you need the ability to answer:
- What exact input did the model receive for prediction X?
- What was the raw feature vector before and after preprocessing?
- What model version was serving at the time of that prediction?
- Was there a recent data pipeline run that could explain a sudden shift?

This requires structured logging at every stage:
- Request ID propagated through the entire pipeline
- Raw input logged at ingestion
- Transformed features logged at the model boundary
- Prediction and confidence logged at output
- Model version and artifact hash in every log record

In distributed systems: distributed tracing (OpenTelemetry) links all these records
under a single trace ID, so you can reconstruct the full journey of any prediction.

## Where it breaks or gets dangerous

**1. Monitoring infrastructure only:**
Many teams instrument their API (latency, error rate) but not their model.
The API runs perfectly. The model quietly degrades.
A healthy infrastructure metric does not imply a healthy model.

**2. No baseline to compare against:**
You set up a dashboard showing feature distributions.
But you never recorded what the training distribution looked like.
You are watching a number with no reference point.
The baseline must be captured at training time and stored alongside the model artifact.

**3. Alert fatigue from poorly calibrated thresholds:**
You set drift alerts too aggressively.
Minor natural variation triggers alerts daily.
On-call team learns to ignore them.
Real drift goes unresponded to.
Threshold calibration is as important as having the alerts.

**4. Ground truth latency problem:**
The model predicts payment delay on invoices.
Ground truth (whether the invoice was actually paid late) is only known 30 days later.
You cannot measure accuracy in real time.
You must design a delayed evaluation pipeline — predict now, evaluate 30 days later.
Many teams skip this because it is engineering-heavy.
Without it, you have no accuracy signal in production.

**5. Logging personal or sensitive data:**
Full observability requires logging raw inputs.
But raw inputs may contain PII — names, account numbers, tax IDs.
In GDPR or MAS TRM contexts, logging PII in observability systems may be a violation.
Log feature vectors, not raw values. Or apply masking before logging.
Observability and data privacy are in direct tension — you have to architect this deliberately.

In [ ]:
# Monitoring and Observability — data drift detection simulation

import numpy as np
from scipy import stats


class ModelMonitor:
    """
    Tracks feature distributions over time.
    At training time: record baseline statistics.
    In production: compare incoming batches against that baseline.
    """

    def __init__(self, feature_names, psi_threshold=0.2, ks_pvalue_threshold=0.05):
        self.feature_names = feature_names
        self.psi_threshold = psi_threshold
        self.ks_pvalue_threshold = ks_pvalue_threshold
        self.baseline = None

    def fit_baseline(self, X_train):
        """Capture training distribution. Must be called once at training time."""
        self.baseline = {}
        for i, name in enumerate(self.feature_names):
            col = X_train[:, i]
            self.baseline[name] = {"mean": np.mean(col), "std": np.std(col), "data": col}
        print("Baseline recorded from training data.")
        for name, stats_dict in self.baseline.items():
            print(f"  {name}: mean={stats_dict['mean']:.3f}, std={stats_dict['std']:.3f}")

    def compute_psi(self, baseline, current, n_bins=10):
        """Population Stability Index: PSI < 0.1 stable, 0.1-0.2 moderate, > 0.2 significant shift."""
        min_val = min(baseline.min(), current.min())
        max_val = max(baseline.max(), current.max())
        bins = np.linspace(min_val, max_val, n_bins + 1)

        base_counts, _ = np.histogram(baseline, bins=bins)
        curr_counts, _ = np.histogram(current, bins=bins)

        base_pct = (base_counts + 1e-6) / len(baseline)
        curr_pct = (curr_counts + 1e-6) / len(current)

        psi = np.sum((curr_pct - base_pct) * np.log(curr_pct / base_pct))
        return psi

    def check_drift(self, X_serving, batch_label="production batch"):
        """Compare a serving batch against the stored baseline."""
        if self.baseline is None:
            raise RuntimeError("Baseline not set. Call fit_baseline first.")

        print(f"\nDrift check: {batch_label}")
        print("-" * 60)
        drift_detected = False

        for i, name in enumerate(self.feature_names):
            col_serving = X_serving[:, i]
            col_baseline = self.baseline[name]["data"]

            ks_stat, ks_pvalue = stats.ks_2samp(col_baseline, col_serving)
            psi = self.compute_psi(col_baseline, col_serving)

            ks_alert = ks_pvalue < self.ks_pvalue_threshold
            psi_alert = psi > self.psi_threshold

            status = "DRIFT" if (ks_alert or psi_alert) else "STABLE"
            if status == "DRIFT":
                drift_detected = True

            print(f"  {name}:")
            print(f"    mean: baseline={col_baseline.mean():.3f}, serving={col_serving.mean():.3f}")
            print(f"    KS p-value: {ks_pvalue:.4f} {'ALERT' if ks_alert else 'ok'}")
            print(f"    PSI: {psi:.4f} {'ALERT' if psi_alert else 'ok'}")
            print(f"    Status: {status}")

        if drift_detected:
            print("\n  ACTION: investigate data pipeline changes. Consider retraining.")
        else:
            print("\n  All features stable.")


np.random.seed(42)

# Training distribution: invoice_amount (log), days_overdue
X_train = np.column_stack([
    np.random.normal(8.0, 1.2, 2000),
    np.random.normal(15.0, 8.0, 2000)
])

monitor = ModelMonitor(feature_names=["log_invoice_amount", "days_overdue"])
monitor.fit_baseline(X_train)

# Week 1: normal traffic
X_week1 = np.column_stack([
    np.random.normal(8.0, 1.2, 500),
    np.random.normal(15.0, 8.0, 500)
])
monitor.check_drift(X_week1, batch_label="Week 1 — expected normal traffic")

# Week 4: upstream data change — invoices now include a different vendor segment
X_week4 = np.column_stack([
    np.random.normal(10.5, 2.1, 500),  # shifted mean: larger invoices
    np.random.normal(35.0, 12.0, 500)  # shifted mean: more overdue
])
monitor.check_drift(X_week4, batch_label="Week 4 — new vendor segment added upstream")

## My D365 analogy

Monitoring a production AI system is structurally identical to subledger reconciliation in D365.

In D365, the general ledger is your source of truth.
Every subledger — accounts payable, accounts receivable, fixed assets, inventory —
must reconcile to the GL at period-end.
You are not checking whether the ledger posting ran. You are checking whether it was correct.
Infrastructure monitoring is checking whether the posting ran.
Model monitoring is checking whether the posting was correct.

The baseline in model monitoring maps to the opening trial balance at period start.
You cannot identify a variance without knowing what the baseline was.
In D365, every reconciliation process begins with a documented opening position.
In ML monitoring, the baseline must be captured at training time and stored.

The ground truth latency problem maps to the accounts receivable aging problem.
You post an invoice today. Whether it is paid on time is only known 30-60 days later.
The AR aging report is the delayed evaluation pipeline:
it compares what was expected (payment terms) against what actually happened (payment date).

Financial controllers run AR aging reports on a schedule.
ML teams should run delayed accuracy reports on the same cadence.
The discipline is identical. The data is different.

## LinkedIn post idea

Financial controllers run subledger reconciliation at period-end.
Not to check whether the posting engine ran.
To check whether the postings were correct.

Most engineering teams monitor their AI APIs the same way they monitor any API.
Latency. Error rate. Uptime.
That tells you whether the engine ran.

It tells you nothing about whether the model is still correct.

A model can have 99.9% uptime and quietly produce wrong predictions
because the data it receives no longer resembles what it was trained on.
No error thrown. No alert fired. Wrong outputs.

Monitoring a production model means tracking the data it receives,
not just the service it runs on.
And it means having a baseline — captured at training time — to compare against.

You cannot reconcile without an opening balance.
You cannot detect drift without a training baseline.

14 years of period-end closes trained me to ask:
not just did it run, but is it right.